In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import pvlib
from pvlib.location import Location

import pytz as pytz

import timezonefinder as tzf

from scipy.interpolate import interp1d
from scipy.optimize import brute
from scipy.integrate import solve_ivp

from tqdm import tqdm

import os

import Module_dynamics as dyn

In [ ]:
#Reimport lib
import importlib
importlib.reload(dyn)
import time

#testing code
capacity = 500
solararea = 0
power = 250

simtime = 60*60*24*1

# Initialize vehicle and environment
path = dyn.Path(name='PROTOUR', type='A2A')
env = dyn.Environment(path, StartDateTimeLocal='2025-06-15 00:00:00')
rider = dyn.Rider()
battery = dyn.Battery(capacity=capacity)
solarpanel = dyn.SolarPanel(area=solararea)
chassis = dyn.Chassis(mass=10, CdA=0.599631, Crr=0.004)
chassis.cargo_mass = 30
motor = dyn.Motor(RatedPower=power, efficiency=1)
vehicle = dyn.Vehicle(env, rider, battery, solarpanel, chassis, motor)
vehicle.environment_filter_alpha = 0.1

#time the loop
t_start = time.time()

# Run simulation (will skip if results file exists)
dyn.simulate_variabletimestep(vehicle, simtime_s=simtime, atol=[1, 0.01, 50.0], rtol=0.001, base_dt=1, max_dv=0.2,
                               max_dt=5, SOC_i=0, output = "full", use_pbar=True, skip_if_result_exist=False)

print(f"Simulation time: {time.time() - t_start} seconds")


In [ ]:
# Post processing one simulation result file
# plot interest variables: position, speed, acceleration, SOC, motor power against time and against distance

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from scipy.optimize import curve_fit
from collections import defaultdict

#set the file to be processed
filepath = 'SIMRESULTS/mass_110.0_motor_250_solar_0.0_battery_500.0_PROTOUR.csv'
data = pd.read_csv(filepath)

time = pd.to_numeric(data['simtime'], errors='coerce').to_numpy()
position = pd.to_numeric(data['Distance'], errors='coerce').to_numpy()
speed = pd.to_numeric(data['Velocity'], errors='coerce').to_numpy()
Energy = pd.to_numeric(data['Energy'], errors='coerce').to_numpy()
motor_power = pd.to_numeric(data['Motorpower'], errors='coerce').to_numpy()
wind_speed_plot = pd.to_numeric(data['WindSpeed'], errors='coerce').to_numpy()

# capacity -> scalar
cap = float(getattr(vehicle.battery, 'capacity', vehicle.battery.capacity))

SOC = Energy / (cap * 3600.0)

# choose time window (seconds)
t_min = 0.0   # set as needed
t_max = 60*60*24*3 # or set custom

mask = (time >= t_min) & (time <= t_max)
time_plot = time[mask]
time_h = time_plot / 3600.0

position_plot = position[mask]
speed_plot = speed[mask]
SOC_plot = SOC[mask]
motor_power_plot = motor_power[mask]
wind_speed_plot = wind_speed_plot[mask]

# quick diagnostics
print("shapes:", time_plot.shape, position_plot.shape, speed_plot.shape, SOC_plot.shape, motor_power_plot.shape)
print("time (s) min/max:", np.nanmin(time_plot), np.nanmax(time_plot))
print("position min/max:", np.nanmin(position_plot), np.nanmax(position_plot))
print("speed min/max:", np.nanmin(speed_plot), np.nanmax(speed_plot))
print("SOC min/max:", np.nanmin(SOC_plot), np.nanmax(SOC_plot))
print("motor power min/max:", np.nanmin(motor_power_plot), np.nanmax(motor_power_plot))


fig, ax1 = plt.subplots(figsize=(12, 6))

# use time in hours for all x
ax1.plot(time_h, position_plot, color='tab:blue', label='Position (m)')
ax1.set_ylabel('Position (m)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(time_h, speed_plot, color='tab:orange', label='Speed (m/s)')
ax2.plot(time_h, wind_speed_plot, color='tab:gray', label='Wind Speed (m/s)', linestyle='dashed')
ax2.set_ylabel('Speed (m/s)', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

ax3 = ax1.twinx()
ax3.spines.right.set_position(("axes", 1.15))
ax3.plot(time_h, SOC_plot, color='tab:green', label='SOC')
ax3.set_ylabel('SOC', color='tab:green')
ax3.tick_params(axis='y', labelcolor='tab:green')
ax3.set_ylim(0.0, 1.0)

ax4 = ax1.twinx()
ax4.spines.right.set_position(("axes", 1.3))
ax4.plot(time_h, motor_power_plot, color='tab:red', label='Motor Power (W)')
ax4.set_ylabel('Motor Power (W)', color='tab:red')
ax4.tick_params(axis='y', labelcolor='tab:red')



# set y-limits: Speed from 0 to max observed speed
ax2.set_ylim(0, max(speed_plot)+1)
# set y-limits: SOC in [0,1]; motor power symmetric around zero
ax3.set_ylim(0.0, 0.1)

# prefer motor rated power if vehicle object exists, otherwise use observed max
try:
    mp_lim = float(vehicle.motor.RatedPower)
except Exception:
    mp_lim = max(1.0, np.nanmax(np.abs(motor_power_plot)))
ax4.set_ylim(-mp_lim-50, mp_lim+50)

# set xlabel in hours and tidy x-ticks (integer hours)
ax1.set_xlabel('Time (h)')
h_min, h_max = np.nanmin(time_h), np.nanmax(time_h)
if np.isfinite(h_min) and np.isfinite(h_max):
    # choose integer tick spacing when span > 1h, else default ticks
    span_h = h_max - h_min
    if span_h > 1.0:
        ticks = np.arange(np.floor(h_min), np.ceil(h_max) + 1, 1.0)
        ax1.set_xticks(ticks)
        ax1.set_xlim(h_min, h_max)

fig.suptitle('Position, Speed, SOC, Motor Power vs Time')
fig.tight_layout()
fig.subplots_adjust(right=0.8)

# combined legend
lines, labels = [], []
for ax in [ax1, ax2, ax3, ax4]:
    l, lab = ax.get_legend_handles_labels()
    lines += l; labels += lab
ax1.legend(lines, labels, loc='upper left')

plt.show()


2025-06-15 24h full day
129.0 kg total mass with 0.5m2 of PV and 500Wh: final distance =  218754.13932084545
avg = 27.329129855
133.0 kg total mass with 1m2 of PV and 500Wh: final distance =  240922.05717548906
avg = 30.115257125

Convergence study on dynamic simulation only (no pv/battery system)
final distance dt 0.1s =  162561.67956226136
final distance dt 0.5s =  162924.8161293678
final distance dt 1s   =  163407.0717180171
final distance dt 2s   =  164233.00078083167
final distance dt 5s   =  168806.842612409
final distance dt 10s  =  180447.25261423894 traces vs time and position do not reflect nicely the cyclic laps of CGV
final distance dt 15s  =  796045.1965056054  traces vs time and position are completely not as expected

In [ ]:
#set the file to be processed
filepath = 'SIMRESULTS/mass_110.0_motor_250_solar_0.0_battery_500.0_PROTOUR.csv'
data = pd.read_csv(filepath)

time = pd.to_numeric(data['simtime'], errors='coerce').to_numpy()
position = pd.to_numeric(data['Distance'], errors='coerce').to_numpy()
speed = pd.to_numeric(data['Velocity'], errors='coerce').to_numpy()
Energy = pd.to_numeric(data['Energy'], errors='coerce').to_numpy()
motor_power = pd.to_numeric(data['Motorpower'], errors='coerce').to_numpy()



# display(data)
plt.plot(data['simtime']/3600, data['WindSpeed'], label='Wind Speed (m/s)')
plt.show()
plt.plot(data['simtime']/3600, data['Solarpower'], label='Solar Power (W)')
plt.show()
plt.plot(data['simtime']/3600, data['slope'], label='slope')
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------
# Load and prepare data
# ------------------------------

filepath = 'SIMRESULTS/mass_110.0_motor_250_solar_0.0_battery_500.0_PROTOUR.csv'
data = pd.read_csv(filepath)

time = pd.to_numeric(data['simtime'], errors='coerce').to_numpy()
position = pd.to_numeric(data['Distance'], errors='coerce').to_numpy()
speed = pd.to_numeric(data['Velocity'], errors='coerce').to_numpy()
Energy = pd.to_numeric(data['Energy'], errors='coerce').to_numpy()
motor_power = pd.to_numeric(data['Motorpower'], errors='coerce').to_numpy()
wind_speed = pd.to_numeric(data['WindSpeed'], errors='coerce').to_numpy()

# battery capacity
cap = float(getattr(vehicle.battery, 'capacity', vehicle.battery.capacity))
SOC = Energy / (cap * 3600.0)

# select window
t_min = 0.0
t_max = 60 * 60 * 24 * 3
mask = (time >= t_min) & (time <= t_max)

time_plot = time[mask]
time_h = time_plot/ 3600.0

position_plot = position[mask]
speed_plot = speed[mask]
SOC_plot = SOC[mask]
motor_power_plot = motor_power[mask]
wind_speed_plot = wind_speed[mask]

# motor power limit
try:
    mp_lim = float(vehicle.motor.RatedPower)
except:
    mp_lim = max(1.0, np.nanmax(np.abs(motor_power_plot)))

# ------------------------------
# Build Plotly figure
# ------------------------------

fig = make_subplots(
    specs=[[{"secondary_y": True}]],
)

# Position
fig.add_trace(
    go.Scatter(x=time_h, y=position_plot, name="Position (m)", line=dict(color="blue")),
    secondary_y=False
)

# Speed
fig.add_trace(
    go.Scatter(x=time_h, y=speed_plot, name="Speed (m/s)", line=dict(color="orange")),
    secondary_y=True
)

# SOC
fig.add_trace(
    go.Scatter(x=time_h, y=SOC_plot, name="SOC", line=dict(color="green")),
    secondary_y=True
)

# Motor power
fig.add_trace(
    go.Scatter(x=time_h, y=motor_power_plot, name="Motor Power (W)", line=dict(color="red")),
    secondary_y=True
)

#Wind speed
fig.add_trace(
    go.Scatter(x=time_h, y=wind_speed_plot, name="Wind Speed (m/s)", line=dict(color="gray", dash='dash')),
    secondary_y=True
)

# ------------------------------
# Layout and axes formatting
# ------------------------------
fig.update_layout(
    title="Position, Speed, SOC, Motor Power vs Time (Interactive)",
    xaxis_title="Time (h)",
    template="plotly_white",
    legend=dict(x=0.01, y=0.99),
    width=1200,
    height=600
)

# left Y axis
fig.update_yaxes(
    title_text="Position (m)",
    secondary_y=False,
)

# right Y axis
fig.update_yaxes(
    title_text="Speed / SOC / Motor Power",
    secondary_y=True,
)

# optional custom limits
fig.update_yaxes(range=[0, max(speed_plot)+1], secondary_y=True)
fig.update_yaxes(range=[0, 1.0], secondary_y=True)
fig.update_yaxes(range=[-mp_lim, mp_lim], secondary_y=True)

fig.show()
